# Monitoring variant: ric/osc_ric/<YOUR_PASSWORD>.ipynb

Based on the upstream example. The original remains under `examples/ric/osc_ric/<YOUR_PASSWORD>.ipynb`.
Core: **oai**; RAN: **oai**. The Open5GS-specific core calculator is disabled for this OAI core. The OAI gNB statistics parser is enabled.

# **5G Deployment Example with: OAI core and RAN (rfsim) and O-RAN SC RIC**

In [ ]:
%pip install -e ../../../.. --extra-index-url=https://gitlab.inria.fr/api/v4/projects/70914/packages/pypi/simple

In [ ]:
from pathlib import Path

Path('input/full').mkdir(parents=True, exist_ok=True)

from metamodel.models.metamodel import MetaModel

metamodel = MetaModel.from_yaml_file("input/metamodel.yaml")
metamodel.to_yaml_file("input/full/metamodel.yaml", exclude_defaults=False)

In [ ]:
from pathlib import Path

Path('input/full').mkdir(parents=True, exist_ok=True)

from orchestrator.models.deployment import DeploymentModel

deployment_model = DeploymentModel.from_yaml_file("input/deployment_model.yaml")
deployment_model.to_yaml_file("input/full/deployment_model.yaml", exclude_defaults=False)

## **1.**   Provisioning:

In [ ]:
from orchestrator.deployment import ProvisioningManagement

prov = ProvisioningManagement(metamodel=metamodel, deployment_model=deployment_model)

In [ ]:
await prov.provision()

### **Checkpoint 1:** *Provisiong finished*

In [ ]:
from pathlib import Path

Path('output/prov').mkdir(parents=True, exist_ok=True)

metamodel, deployment_model = prov.output()

metamodel.to_yaml_file("output/prov/metamodel.yaml")
deployment_model.to_yaml_file("output/prov/deployment_model.yaml")

## **2.**   Kubernetes deployement:

In [ ]:
from metamodel.models.metamodel import MetaModel

from orchestrator.models.deployment import DeploymentModel

deployment_model = DeploymentModel.from_yaml_file("output/prov/deployment_model.yaml")
metamodel = MetaModel.from_yaml_file("output/prov/metamodel.yaml")

In [ ]:
from orchestrator.deployment import K8SManagement

k8s = K8SManagement(metamodel=metamodel, deployment_model=deployment_model)

In [ ]:
await k8s.deploy()

### **Checkpoint 2:** *Kubernetes deployed*

In [ ]:
from pathlib import Path

Path('output/k8s').mkdir(parents=True, exist_ok=True)

metamodel, deployment_model = k8s.output()

metamodel.to_yaml_file("output/k8s/metamodel.yaml")
deployment_model.to_yaml_file("output/k8s/deployment_model.yaml")

## **3.**   5G core deployement:

In [ ]:
from metamodel.models.metamodel import MetaModel

from orchestrator.models.deployment import DeploymentModel

deployment_model = DeploymentModel.from_yaml_file("output/k8s/deployment_model.yaml")
metamodel = MetaModel.from_yaml_file("output/k8s/metamodel.yaml")

In [ ]:
from orchestrator.deployment import CoreManagement

core = CoreManagement(metamodel=metamodel, deployment_model=deployment_model)

In [ ]:
await core.deploy()

### **Checkpoint 3:** *5G core deployed*

In [ ]:
from pathlib import Path

Path('output/core').mkdir(parents=True, exist_ok=True)

metamodel, deployment_model = core.output()

metamodel.to_yaml_file("output/core/metamodel.yaml")
deployment_model.to_yaml_file("output/core/deployment_model.yaml")

## **4.**   RIC deployement:

In [ ]:
from metamodel.models.metamodel import MetaModel

from orchestrator.models.deployment import DeploymentModel

deployment_model = DeploymentModel.from_yaml_file("output/core/deployment_model.yaml")
metamodel = MetaModel.from_yaml_file("output/core/metamodel.yaml")

In [ ]:
from orchestrator.deployment import RICManagement

ric = RICManagement(metamodel=metamodel, deployment_model=deployment_model)

In [ ]:
await ric.deploy()

### **Checkpoint 4:** *RIC deployed*

In [ ]:
from pathlib import Path

Path('output/ric').mkdir(parents=True, exist_ok=True)

metamodel, deployment_model = ric.output()

metamodel.to_yaml_file("output/ric/metamodel.yaml")
deployment_model.to_yaml_file("output/ric/deployment_model.yaml")

## **5.**   RAN deployement:

In [ ]:
from metamodel.models.metamodel import MetaModel

from orchestrator.models.deployment import DeploymentModel

deployment_model = DeploymentModel.from_yaml_file("output/ric/deployment_model.yaml")
metamodel = MetaModel.from_yaml_file("output/ric/metamodel.yaml")

In [ ]:
from orchestrator.deployment import RANManagement

ran = RANManagement(metamodel=metamodel, deployment_model=deployment_model)

In [ ]:
# Enable the statistics-file exporter before the gNB pod is created.
from orchestrator.models.ran import OAIAdvancedGNBConfig

gnb = ran.deployment_model.ran.components["gnb"]
gnb.advanced_config = OAIAdvancedGNBConfig.model_validate(
    gnb.advanced_config.model_dump() if gnb.advanced_config is not None else {}
)
gnb.advanced_config.metrics_parser.enabled = True

In [ ]:
await ran.deploy()

### **Checkpoint 5:** *RAN deployed*

In [ ]:
from pathlib import Path

Path('output/ran').mkdir(parents=True, exist_ok=True)

metamodel, deployment_model = ran.output()

metamodel.to_yaml_file("output/ran/metamodel.yaml")
deployment_model.to_yaml_file("output/ran/deployment_model.yaml")

## **6.**   UE deployement:

In [ ]:
from metamodel.models.metamodel import MetaModel

from orchestrator.models.deployment import DeploymentModel

deployment_model = DeploymentModel.from_yaml_file("output/ran/deployment_model.yaml")
metamodel = MetaModel.from_yaml_file("output/ran/metamodel.yaml")

## Monitoring before UE attachment

The profile uses the control host selected by Kubernetes, including physical
Sopnodes. Capture interfaces refer to pod interfaces, not the host or radio NIC.
Keep the existing provisioning, RF and Multus settings. Grafana uses `admin` / `admin`.

In [ ]:
from pathlib import Path
import asyncio
import json

from orchestrator.deployment import MonitoringManagement
from orchestrator.models.monitoring import MonitoringDeployment

QUERIES_FILE = Path("queries.json")  # Your Prometheus export query profile.

monitoring = MonitoringManagement(preceding=ran)
monitoring.deployment_model.monitoring = MonitoringDeployment.from_yaml_file("monitoring.yaml")

In [ ]:
try:
    await monitoring.deploy()
finally:
    monitoring.output(
        metamodel_file="monitoring_metamodel.yaml",
        deployment_model_file="monitoring_deployment_model.yaml",
    )

### Open Grafana

The SSH tunnel uses the current model address and jump host.
Open the link after deployment; close the tunnel with `grafana_tunnel.terminate()`.

In [ ]:
import atexit
import shlex
import subprocess
from IPython.display import Markdown, display

# Close the previous tunnel before opening one with the current host address.
if "grafana_tunnel" in globals() and grafana_tunnel.poll() is None:
    grafana_tunnel.terminate()
    await asyncio.to_thread(grafana_tunnel.wait, 5)

# get SSH info and fallback
cfg = monitoring.deployment_model.monitoring
control = cfg.installed_control_node or cfg.control_node or monitoring.deployment_model.params.k8s.master_node
ssh = monitoring.deployment_model.nodes[control].ssh_info

command = ["ssh", "-N", "-T", "-S", "none", "-o", "ControlMaster=no", "-o", "ControlPersist=no",
           "-o", "BatchMode=yes", "-o", "ExitOnForwardFailure=yes", "-o", "ConnectTimeout=10"]

command += shlex.split(ssh.common_args or "")

if ssh.private_key_file:
    command += ["-i", str(Path(ssh.private_key_file).expanduser())]
command += ["-p", str(ssh.port), "-L", f"127.0.0.1:30096:127.0.0.1:{cfg.grafana_node_port}",
            f"{ssh.user}@{ssh.hostname or control}"]

grafana_tunnel = subprocess.Popen(command, stdout=subprocess.DEVNULL)

# Close the tunnel when the kernel exits normally.
atexit.register(grafana_tunnel.terminate)

GRAFANA_URL = "http://127.0.0.1:30096"
display(Markdown(f"[Open Grafana]({GRAFANA_URL}) — username: **admin**, password: **admin**"))

In [ ]:
from orchestrator.deployment import UEManagement

ue = UEManagement(preceding=monitoring)


In [ ]:
await ue.deploy()

### **Checkpoint 6:** *UE deployed*

## Iperf with experiment artifacts

The server runs on the Kubernetes control host from the deployment model.
Each client binds to its own 5G interface. Review the explicit scenario before running it.
The collector exports Prometheus data after traffic and saves each report with the timeline.

In [ ]:
from datetime import UTC, datetime
from orchestrator.artifacts import ExperimentRun
from orchestrator.deployment import ProvisioningManagement
from orchestrator.utils.ansible import get_ansible_inventory_line, run_ansible_command, run_ansible_playbook

SERVER_NODE = monitoring.deployment_model.params.k8s.master_node
CLIENTS = ['sim_ue']
UE_INTERFACE = 'oaitun_ue1'
DURATION, BASE_PORT = 20, 5202
BASELINE_SECONDS, COLLECTION_SECONDS = 5, 5

server_node = monitoring.deployment_model.nodes[SERVER_NODE]
server_ip = server_node.ssh_info.hostname or SERVER_NODE
server_inventory = get_ansible_inventory_line(SERVER_NODE, server_node)
server_runner = str(Path(monitoring.deployment_model.params.ansible.runner_path) / "iperf-server")
clients = ue

await run_ansible_playbook(
    playbook=[{
        "name": "Install iperf3 on the server host", "hosts": SERVER_NODE,
        "gather_facts": False, "become": True,
        "tasks": [{"ansible.builtin.apt": {"name": "iperf3", "state": "present",
                  "update_cache": True, "cache_valid_time": 3600}}],
    }],
    inventory=server_inventory, runner_path=server_runner, quiet=True,
)
await clients.run_command(
    command=["sh", "-c", "DEBIAN_FRONTEND=noninteractive apt-get update && DEBIAN_FRONTEND=noninteractive apt-get install -y iperf3"],
    components=CLIENTS, quiet=True,
)
addresses = await clients.run_command(
    command=["ip", "-j", "-4", "addr", "show", "dev", UE_INTERFACE],
    components=CLIENTS, quiet=True,
)
ue_ips = {}
for client in CLIENTS:
    ipv4 = [a["local"] for device in json.loads(addresses[client]) for a in device["addr_info"]
            if a["family"] == "inet"]
    if len(ipv4) != 1:
        raise RuntimeError(f"{client}: expected one IPv4 address on {UE_INTERFACE}")
    ue_ips[client] = ipv4[0]
print(f"Server: {SERVER_NODE} ({server_ip}); UE clients: {ue_ips}")

In [ ]:
async def start_iperf_server(port):
    await run_ansible_command(
        command=["iperf3", "-s", "-D", "-1", "--idle-timeout", "60",
                 "-B", server_ip, "-p", str(port), "-J"],
        hosts=SERVER_NODE, inventory=server_inventory, runner_path=server_runner, quiet=True,
    )


async def iperf_client(client, direction, duration, port):
    reverse = {"uplink": [], "downlink": ["-R"]}[direction]
    output = await clients.run_command(
        command=["iperf3", "-c", server_ip, "-B", ue_ips[client], "--bind-dev", UE_INTERFACE,
                 "-p", str(port), "-t", str(duration), "--connect-timeout", "10000",
                 "-J", "--get-server-output", *reverse],
        components=[client], quiet=True,
    )
    report = json.loads(output[client])
    if report.get("error"):
        raise RuntimeError(report["error"])
    return report

In [ ]:
SCENARIO = [
    {"name": "ue1_uplink", "client": 'sim_ue', "direction": "uplink", "duration": DURATION},
    {"name": "ue1_downlink", "client": 'sim_ue', "direction": "downlink", "duration": DURATION},
]

In [ ]:
timestamp = datetime.now(UTC).strftime("%Y-%m-%d_%H-%M-%S.%fZ")
output_directory = Path("results") / f"iperf-{timestamp}"

async with ExperimentRun(
    output_directory,
    name="ue-to-host-iperf",
    monitoring=monitoring,
    queries_file=QUERIES_FILE,
) as experiment:
    experiment.save_json("scenario.json", SCENARIO)

    with experiment.section("baseline"):
        await asyncio.sleep(BASELINE_SECONDS)

    for port, <YOUR_PASSWORD> in enumerate(SCENARIO, start=BASE_PORT):
        print(f"Running {<YOUR_PASSWORD>['name']}: {<YOUR_PASSWORD>['client']}, {<YOUR_PASSWORD>['direction']}, {<YOUR_PASSWORD>['duration']} seconds")
        await start_iperf_server(port)

        with experiment.section(<YOUR_PASSWORD>["name"]):
            report = await iperf_client(
                client=<YOUR_PASSWORD>["client"],
                direction=<YOUR_PASSWORD>["direction"],
                duration=<YOUR_PASSWORD>["duration"],
                port=port,
            )

        experiment.save_json(f"{<YOUR_PASSWORD>['name']}.json", report)

    with experiment.section("collection"):
        await asyncio.sleep(COLLECTION_SECONDS)

print(f"Artifacts saved to: {output_directory.resolve()}")

In [ ]:
from pathlib import Path

Path('output/ue').mkdir(parents=True, exist_ok=True)

metamodel, deployment_model = ue.output()

metamodel.to_yaml_file("output/ue/metamodel.yaml")
deployment_model.to_yaml_file("output/ue/deployment_model.yaml")

## Cleanup

- #### UE cleanup:

In [ ]:
# from orchestrator.deployment import UEManagement
# from metamodel.models.metamodel import MetaModel
# from orchestrator.models.deployment import DeploymentModel

# deployment_model = DeploymentModel.from_yaml_file("output/ue/deployment_model.yaml")
# metamodel = MetaModel.from_yaml_file("output/ue/metamodel.yaml")

# ue = UEManagement(metamodel=metamodel, deployment_model=deployment_model)

# await ue.remove(force=True)

- #### RIC cleanup:

In [ ]:
# from orchestrator.deployment import RICManagement
# from metamodel.models.metamodel import MetaModel
# from orchestrator.models.deployment import DeploymentModel

# deployment_model = DeploymentModel.from_yaml_file("output/ric/deployment_model.yaml")
# metamodel = MetaModel.from_yaml_file("output/ric/metamodel.yaml")

# ric = RICManagement(metamodel=metamodel, deployment_model=deployment_model)

# await ric.remove(force=True)

- #### RAN cleanup:

In [ ]:
# from orchestrator.deployment import RANManagement
# from metamodel.models.metamodel import MetaModel
# from orchestrator.models.deployment import DeploymentModel

# deployment_model = DeploymentModel.from_yaml_file("output/ran/deployment_model.yaml")
# metamodel = MetaModel.from_yaml_file("output/ran/metamodel.yaml")

# ran = RANManagement(metamodel=metamodel, deployment_model=deployment_model)

# await ran.remove(force=True)

- #### 5G core cleanup:

In [ ]:
# from orchestrator.deployment import CoreManagement
# from metamodel.models.metamodel import MetaModel
# from orchestrator.models.deployment import DeploymentModel

# deployment_model = DeploymentModel.from_yaml_file("output/core/deployment_model.yaml")
# metamodel = MetaModel.from_yaml_file("output/core/metamodel.yaml")

# core = CoreManagement(metamodel=metamodel, deployment_model=deployment_model)

# await core.remove(force=True)

- #### Kubernetes cleanup:

In [ ]:
# from orchestrator.deployment import K8SManagement
# from metamodel.models.metamodel import MetaModel
# from orchestrator.models.deployment import DeploymentModel

# deployment_model = DeploymentModel.from_yaml_file("output/k8s/deployment_model.yaml")
# metamodel = MetaModel.from_yaml_file("output/k8s/metamodel.yaml")

# k8s = K8SManagement(metamodel=metamodel, deployment_model=deployment_model)

# await k8s.remove(force=True)

- #### Provisioning cleanup:

In [ ]:
# from orchestrator.deployment import ProvisioningManagement
# from metamodel.models.metamodel import MetaModel
# from orchestrator.models.deployment import DeploymentModel

# deployment_model = DeploymentModel.from_yaml_file("output/prov/deployment_model.yaml")
# metamodel = MetaModel.from_yaml_file("output/prov/metamodel.yaml")

# prov = ProvisioningManagement(metamodel=metamodel, deployment_model=deployment_model)

# await prov.unprovision(force=True)